# Sprint 14 — SQL: o mercado de apps de leitura

### por Luiz Trajano

---

## O que é este documento

Com a pandemia, as pessoas passaram mais tempo em casa e começaram a ler mais. Várias startups lançaram aplicativos para quem gosta de ler, e eu recebi o banco de dados de um desses serviços. Ele tem livros, autores, editoras, as notas que os usuários deram aos livros e as resenhas que escreveram.

Uso SQL para responder cinco perguntas sobre esse banco. As respostas servem de base para a proposta de valor de um novo produto nesse mercado. O notebook segue esta ordem: a conexão com o banco, as tabelas e como elas se ligam, uma consulta por tarefa com a conclusão logo abaixo e, no fim, a conclusão geral.

Neste notebook, "nota" é a tabela `ratings` e "resenha" é o texto da tabela `reviews`.

> Este notebook roda só na plataforma da TripleTen, onde o banco está acessível. Na cópia local, a senha fica como `<SENHA>`.

## Índice

- [0. Conexão com o banco](#0.-Conexão-com-o-banco)
- [1. Objetivo do estudo](#1.-Objetivo-do-estudo)
- [2. As tabelas e como se ligam](#2.-As-tabelas-e-como-se-ligam)
- [3. Tarefas](#3.-Tarefas)
- [4. Conclusão geral](#4.-Conclusão-geral)

---

# 0. Conexão com o banco

Para me conectar ao banco, uso o código de conexão fornecido pela TripleTen. Mudei uma coisa só: corrigi um erro de digitação do material original (`db_confi` virou `db_config`), que fazia a célula falhar. A senha só existe na versão da plataforma. A conexão fica guardada na variável `engine`.

Depois crio a função `consulta`, que recebe o texto de uma consulta SQL e devolve o resultado como DataFrame. Assim cada tarefa cabe em uma chamada, e não repito o `read_sql` pelo notebook inteiro. Toda conta (contagem, média, filtro, ordenação e junção) acontece dentro do SQL. O pandas só mostra e guarda o resultado.

Para testar a conexão, consulto o catálogo do banco. Ele é compartilhado com outros projetos da TripleTen e tem mais tabelas do que este estudo usa, então filtro só as cinco do enunciado (`books`, `authors`, `publishers`, `ratings` e `reviews`) e confirmo que todas existem.

In [103]:
# import libraries
import pandas as pd
from sqlalchemy import create_engine


db_config = {
                'user': 'practicum_student', # username
                'pwd': '<SENHA>', # password
                'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
                'port': 5432, # connection port
                'db': 'data-analyst-final-project-db' # the name of the database
            }
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                         db_config['pwd'],
                                                         db_config['host'],
                                                         db_config['port'],
                                                         db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [104]:
def consulta(sql):
    """Executa uma consulta SQL no banco e devolve o resultado como DataFrame.

    Parâmetros
    ----------
    sql : texto da consulta SQL.

    Retorna
    -------
    DataFrame : o resultado da consulta, só para exibir (nenhum cálculo é feito em pandas).
    """
    # read_sql manda o texto ao PostgreSQL pela engine; o banco calcula, o pandas só recebe a tabela pronta
    return pd.io.sql.read_sql(sql, con=engine)

In [105]:
# as 5 tabelas do enunciado; a mesma lista alimenta o laço da seção 2.1
TABELAS = ['books', 'authors', 'publishers', 'ratings', 'reviews']

# o banco é compartilhado com outros projetos
# assim que verifiquei e abri na minha máquina
# vi 12 tabelas, então o WHERE ... IN filtra só as tabelas deste estudo
tabelas = consulta(f"""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    AND table_name IN ('{"', '".join(TABELAS)}')
    ORDER BY table_name;
""")
tabelas

,table_name
0,authors
1,books
2,publishers
3,ratings
4,reviews


---

# 1. Objetivo do estudo

> **Enunciado:** "Descreva os objetivos do estudo."

Quero entender o que o catálogo e os usuários deste serviço mostram, para apoiar a proposta de valor de um novo aplicativo de leitura. Cada tarefa responde a uma pergunta que entra nessa proposta:

1. Quantos livros foram publicados depois de 01/01/2000 (seção 3.1). Mostra se o acervo é recente ou antigo, o que conta para um app que quer oferecer lançamentos.
2. Quantas resenhas e qual a nota média de cada livro (seção 3.2). Mostra quais livros os leitores avaliam bem e quanto eles se envolvem com cada título.
3. Qual editora tem mais livros com mais de 50 páginas (seção 3.3). O filtro tira os folhetos e aponta a editora com mais títulos no catálogo, candidata a uma parceria.
4. Qual autor tem a maior nota média, contando só livros com 50 notas ou mais (seção 3.4). O mínimo de 50 notas evita que um livro com poucas opiniões puxe o resultado.
5. Quantas resenhas, em média, escrevem os usuários que deram nota a mais de 50 livros (seção 3.5). Mostra se os leitores mais ativos também escrevem, o que ajuda a decidir se vale investir em resenhas e comunidade no app.

---

# 2. As tabelas e como se ligam

> **Enunciado:** "Estude as tabelas (imprima as primeiras linhas)" e explique a relação entre elas no modelo de dados.

## 2.1 Primeiras linhas e tamanho de cada tabela

Olho as cinco tabelas antes de qualquer tarefa, para saber o que cada linha representa e quais colunas ligam uma tabela à outra. Crio a função `primeiras_linhas` e chamo ela num laço pela lista `TABELAS` (seção 0), assim não repito o mesmo bloco cinco vezes. Depois conto as linhas de cada tabela numa consulta só: um `COUNT(*)` por tabela, empilhados com `UNION ALL`, para ver os tamanhos lado a lado.

In [106]:
def primeiras_linhas(tabela, n=5):
    """Busca as primeiras linhas de uma tabela do banco.

    Parâmetros
    ----------
    tabela : nome da tabela no banco.
    n : quantas linhas trazer (padrão 5).

    Retorna
    -------
    DataFrame : as n primeiras linhas da tabela.
    """
    # o LIMIT corta no banco, então só n linhas chegam ao notebook
    return consulta(f'SELECT * FROM {tabela} LIMIT {n};')


# display dentro do laço mostra as 5 tabelas; sem ele, só a última apareceria
for tabela in TABELAS:
    print(f'Tabela: {tabela}')
    display(primeiras_linhas(tabela))

Tabela: books


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Tabela: authors


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Tabela: publishers


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


Tabela: ratings


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Tabela: reviews


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


In [107]:
# um SELECT por tabela, empilhados com UNION ALL: o banco devolve os 5 tamanhos numa tabela só
sql_tamanhos = '\nUNION ALL\n'.join(
    f"SELECT '{tabela}' AS tabela, COUNT(*) AS linhas FROM {tabela}" for tabela in TABELAS
)

# o ORDER BY depois do último UNION ALL ordena o resultado inteiro
tamanhos = consulta(sql_tamanhos + '\nORDER BY linhas DESC;')
tamanhos

,tabela,linhas
0,ratings,6456
1,reviews,2793
2,books,1000
3,authors,636
4,publishers,340


**Conclusão 2.1 — O banco tem 1.000 livros, 6.456 notas e 2.793 resenhas, e cada tabela guarda um tipo de registro:**

**O que é uma linha em cada tabela.** Em `books`, uma linha é um livro, com autor, editora, número de páginas e data de publicação (1.000 linhas). Em `authors` (636) e `publishers` (340), uma linha é um autor ou uma editora, só com o id e o nome. Em `ratings` (6.456), uma linha é a nota que um usuário deu a um livro. Em `reviews` (2.793), uma linha é uma resenha com texto.

**Tamanhos.** Há mais livros do que autores, então alguns autores têm mais de um livro no catálogo. As notas são a maior tabela, com mais do que o dobro das resenhas. Minha leitura é que dar nota custa menos esforço do que escrever, por isso há mais notas do que resenhas.

**O que as primeiras linhas já mostram.** A mesma editora aparece com nomes diferentes (`Ace`, `Ace Book`, `Ace Books`, `Ace Hardcover`), cada um com o seu `publisher_id`. Na tarefa 3.3, a contagem por editora separa essas variações, e lá decido se agrupo ou não. Um `author_id` também pode juntar coautores num nome só (`Aesop/Laura Harris/Laura Gibbs`), então na tarefa 3.4 o autor é esse registro, não cada pessoa, definindo assim por trabalhos em conjunto. Por fim, o texto das resenhas nas primeiras linhas é gerado, com frases sem sentido como "Mention society tell send professor analysis". Ele serve para contar resenhas, não para ler a opinião dos usuários.

## 2.2 Modelo de dados

Pelas colunas, `books` é a tabela central. Cada livro aponta para um autor (`author_id` em `authors`) e para uma editora (`publisher_id` em `publishers`). As notas (`ratings`) e as resenhas (`reviews`) apontam para o livro pelo `book_id`, e a coluna `username` liga a nota e a resenha do mesmo usuário, que é o que a tarefa 3.5 usará.

Antes das tarefas, confiro três coisas com SQL:

1. Se toda chave encontra par na tabela de origem. Uma chave sem par faz o `JOIN` perder linhas sem aviso. Uso `LEFT JOIN` com `IS NULL` para contar as que ficam sem par, mesmo padrão de checagem que faria em python, só que checando em SQL.
2. Quantas notas e resenhas cada livro tem. Conto cada tabela por livro numa CTE separada antes de juntar com `books`.
3. Quanto o `JOIN` direto de `ratings` com `reviews` multiplica as linhas. Cada nota de um livro se combina com cada resenha do mesmo livro, e é por isso que, na tarefa 3.2, agrego as duas tabelas antes de juntar.

In [108]:
# LEFT JOIN + IS NULL conta as chaves sem par na tabela de origem; 
# zero em todas = ligações íntegras
sem_par = consulta("""
    SELECT 'books.author_id -> authors' AS ligacao, COUNT(*) AS sem_par
    FROM books AS books
    LEFT JOIN authors AS authors ON authors.author_id = books.author_id
    WHERE authors.author_id IS NULL

    UNION ALL

    SELECT 'books.publisher_id -> publishers', COUNT(*)
    FROM books AS books
    LEFT JOIN publishers AS publishers ON publishers.publisher_id = books.publisher_id
    WHERE publishers.publisher_id IS NULL

    UNION ALL

    SELECT 'ratings.book_id -> books', COUNT(*)
    FROM ratings AS ratings
    LEFT JOIN books AS books ON books.book_id = ratings.book_id
    WHERE books.book_id IS NULL

    UNION ALL

    SELECT 'reviews.book_id -> books', COUNT(*)
    FROM reviews AS reviews
    LEFT JOIN books AS books ON books.book_id = reviews.book_id
    WHERE books.book_id IS NULL;
""")
sem_par

,ligacao,sem_par
0,books.author_id -> authors,0
1,books.publisher_id -> publishers,0
2,ratings.book_id -> books,0
3,reviews.book_id -> books,0


In [109]:
# cada CTE conta uma tabela por livro antes do JOIN, então uma não multiplica as linhas da outra
# pesquisei como incluir na média o livro sem resenha e aprendi/li sobre:
# o COALESCE: ele troca NULL por 0 (semelhante ao fillna do pandas)
# sem ele, o AVG ignora o NULL e calcula a média só entre os livros com resenha, que sai mais alta
por_livro = consulta("""
    WITH notas AS (
        SELECT book_id, COUNT(*) AS n_notas
        FROM ratings
        GROUP BY book_id
    ),
    resenhas AS (
        SELECT book_id, COUNT(*) AS n_resenhas
        FROM reviews
        GROUP BY book_id
    )
    SELECT
        COUNT(*)                                        AS livros,
        COUNT(notas.book_id)                            AS livros_com_nota,
        COUNT(resenhas.book_id)                         AS livros_com_resenha,
        ROUND(AVG(COALESCE(notas.n_notas, 0)), 2)       AS media_notas_por_livro,
        MAX(notas.n_notas)                              AS max_notas,
        ROUND(AVG(COALESCE(resenhas.n_resenhas, 0)), 2) AS media_resenhas_por_livro,
        MAX(resenhas.n_resenhas)                        AS max_resenhas
    FROM books AS books
    LEFT JOIN notas AS notas ON notas.book_id = books.book_id
    LEFT JOIN resenhas AS resenhas ON resenhas.book_id = books.book_id;
""")
por_livro

,livros,livros_com_nota,livros_com_resenha,media_notas_por_livro,max_notas,media_resenhas_por_livro,max_resenhas
0,1000,1000,994,6.46,160,2.79,7


In [110]:
# prova da armadilha: o JOIN direto combina cada nota com cada resenha do mesmo livro
multiplicacao = consulta("""
    SELECT
        (SELECT COUNT(*) FROM ratings) AS linhas_ratings,
        (SELECT COUNT(*) FROM reviews) AS linhas_reviews,
        (SELECT COUNT(*)
        FROM ratings AS ratings
        JOIN reviews AS reviews ON reviews.book_id = ratings.book_id) AS linhas_join_direto;
""")
multiplicacao

,linhas_ratings,linhas_reviews,linhas_join_direto
0,6456,2793,26167


**Conclusão 2.2 — `books` é a tabela central, todas as chaves têm par, e `ratings` e `reviews` precisam ser agregadas antes de qualquer junção:**

**Como as tabelas se ligam.** `books` liga cada livro a um autor (`author_id`) e a uma editora (`publisher_id`). `ratings` e `reviews` ligam as notas e as resenhas ao livro pelo `book_id`. Um autor ou uma editora pode ter vários livros, e um livro pode ter várias notas e várias resenhas, então todas as ligações são de um para muitos. A coluna `username` aparece em `ratings` e em `reviews` e liga a nota e a resenha do mesmo usuário, que é o que a tarefa 3.5 usa.

**As chaves estão íntegras.** Nenhuma das quatro ligações tem chave sem par: 0 livros sem autor, 0 livros sem editora, 0 notas e 0 resenhas apontando para um livro que não existe. Um JOIN entre essas tabelas não perde linha por falta de correspondência.

**Cobertura.** Todos os 1.000 livros têm pelo menos uma nota, e 994 (99,4%) têm pelo menos uma resenha. Os 6 livros sem resenha são o motivo do `LEFT JOIN` e do `COALESCE` na tarefa 3.2: com um JOIN comum, eles sumiriam do resultado. Em média, cada livro tem 6,46 notas (máximo de 160) e 2,79 resenhas (máximo de 7).

**A armadilha da junção.** Juntar `ratings` e `reviews` direto pelo `book_id` gera 26.167 linhas, contra 6.456 notas e 2.793 resenhas, porque cada nota se combina com cada resenha do mesmo livro. Nessa tabela, a contagem de resenhas sai inflada e os livros com mais resenhas pesam mais na média geral das notas. Por isso, na tarefa 3.2, agrego cada tabela por livro antes de juntar com `books`.

---

# 3. Tarefas

> **Enunciado:** "Faça uma consulta SQL para cada uma das tarefas, enuncie os resultados e descreva suas conclusões."

## 3.1 Livros publicados após 01/01/2000

> *Enunciado: "Encontre o número de livros publicados após 1º de janeiro de 2000."*

Conto os livros com `publication_date` depois de 01/01/2000. O enunciado diz "após", então uso `>` e deixo de fora quem saiu exatamente nesse dia. Para mostrar que essa escolha não esconde nada, conto à parte os livros publicados em 01/01/2000.

Na mesma consulta trago o total do catálogo, a parcela em porcentagem, os livros sem data (que não entram em nenhum dos lados) e as datas mais antiga e mais recente, para situar o acervo no tempo. Uso `FILTER (WHERE ...)`, que faz o `COUNT` contar só as linhas que passam na condição. Assim todas as contagens saem de uma consulta só.

In [111]:
# o enunciado diz "após": uso > e conto à parte quem saiu exatamente nessa data
DATA_CORTE = '2000-01-01'

# FILTER (WHERE ...) faz cada COUNT contar só as linhas que passam na condição
# COUNT(*) - COUNT(publication_date) dá os livros sem data, porque COUNT(coluna) ignora NULL
livros_2000 = consulta(f"""
    SELECT
        COUNT(*) FILTER (WHERE publication_date > '{DATA_CORTE}') AS livros_apos_corte,
        COUNT(*) FILTER (WHERE publication_date = '{DATA_CORTE}') AS livros_na_data_corte,
        COUNT(*)                                                  AS total_livros,
        ROUND(100.0 * COUNT(*) FILTER (WHERE publication_date > '{DATA_CORTE}') / COUNT(*), 1) AS pct_apos_corte,
        COUNT(*) - COUNT(publication_date)                        AS livros_sem_data,
        MIN(publication_date)                                     AS data_mais_antiga,
        MAX(publication_date)                                     AS data_mais_recente
    FROM books;
""")
display(livros_2000)

,livros_apos_corte,livros_na_data_corte,total_livros,pct_apos_corte,livros_sem_data,data_mais_antiga,data_mais_recente
0,819,2,1000,81.9,0,1952-12-01,2020-03-31


**Conclusão 3.1 — 819 dos 1.000 livros (81,9%) foram publicados após 01/01/2000:**

**O corte.** Uso `>` porque o enunciado diz "após". Só 2 livros saíram exatamente em 01/01/2000, então incluí-los mudaria pouco o resultado. Dos outros 181, 179 são anteriores a 2000.

**O acervo no tempo.** Nenhum livro está sem data, e as publicações vão de 01/12/1952 a 31/03/2020. A maior parte do catálogo foi publicada após 01/01/2000.

**Para o novo produto.** O catálogo do concorrente é majoritariamente de livros publicados nas últimas duas décadas antes dos dados. Um app que queira competir com ele precisa de um acervo atual, e os livros mais antigos são uma parcela pequena do que os leitores encontram hoje nesse serviço.

## 3.2 Número de resenhas e nota média por livro

> *Enunciado: "Encontre o número de resenhas de usuários e a nota média para cada livro."*

Para cada livro, conto as resenhas em `reviews` e calculo a nota média em `ratings`. Como vi na seção 2.2, juntar as duas tabelas direto multiplica as linhas, então agrego cada uma por livro numa CTE e só depois junto com `books`. Uso `LEFT JOIN` para manter os 6 livros sem resenha, com `COALESCE` pondo 0 no número de resenhas deles. Também trago o número de notas de cada livro, porque uma média calculada com 2 notas pesa menos do que uma calculada com 100.

O resultado tem uma linha por livro e é difícil de ler inteiro. Por isso faço uma segunda consulta que resume essa tabela: a média e os extremos da nota média, quantos livros têm nota média 5 e com quantas notas, e a correlação entre o número de resenhas e a nota média (`CORR`, função do PostgreSQL), para ver se o livro mais resenhado tem nota melhor.

In [112]:
# cada CTE agrega uma tabela por livro antes do JOIN (seção 2.2: o JOIN direto multiplica as linhas)
# guardo o texto numa variável porque a consulta de resumo logo abaixo reaproveita ele
sql_livros = """
    WITH resenhas AS (
        SELECT book_id, COUNT(*) AS n_resenhas
        FROM reviews
        GROUP BY book_id
    ),
    notas AS (
        SELECT book_id, COUNT(*) AS n_notas, ROUND(AVG(rating), 2) AS nota_media
        FROM ratings
        GROUP BY book_id
    )
    SELECT
        books.book_id,
        books.title,
        COALESCE(resenhas.n_resenhas, 0) AS n_resenhas,
        notas.n_notas,
        notas.nota_media
    FROM books AS books
    LEFT JOIN resenhas AS resenhas ON resenhas.book_id = books.book_id
    LEFT JOIN notas AS notas ON notas.book_id = books.book_id
"""

# LEFT JOIN mantém os livros sem resenha; o ORDER BY põe os mais resenhados no topo
resenhas_e_notas = consulta(sql_livros + 'ORDER BY n_resenhas DESC, nota_media DESC;')
resenhas_e_notas

,book_id,title,n_resenhas,n_notas,nota_media
0,948,Twilight (Twilight #1),7,160,3.66
1,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,82,4.41
2,299,Harry Potter and the Chamber of Secrets (Harry...,6,80,4.29
3,656,The Book Thief,6,53,4.26
4,734,The Glass Castle,6,29,4.21
...,...,...,...,...,...
995,387,Leonardo's Notebooks,0,2,4.00
996,191,Disney's Beauty and the Beast (A Little Golden...,0,1,4.00
997,221,Essential Tales and Poems,0,3,4.00
998,83,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3,3.67


In [113]:
# a tabela por livro vira uma CTE, e o resumo é calculado em cima dela, ainda no banco
# CORR é a correlação de Pearson do PostgreSQL; o ::numeric converte o resultado para o ROUND aceitar
resumo_livros = consulta(f"""
    WITH por_livro AS ({sql_livros})
    SELECT
        COUNT(*)                                        AS livros,
        ROUND(AVG(nota_media), 2)                       AS media_das_notas_medias,
        MIN(nota_media)                                 AS menor_nota_media,
        MAX(nota_media)                                 AS maior_nota_media,
        COUNT(*) FILTER (WHERE nota_media = 5)          AS livros_nota_5,
        MAX(n_notas) FILTER (WHERE nota_media = 5)      AS max_notas_entre_nota_5,
        ROUND(CORR(n_resenhas, nota_media)::numeric, 2) AS corr_resenhas_nota
    FROM por_livro;
""")
resumo_livros

,livros,media_das_notas_medias,menor_nota_media,maior_nota_media,livros_nota_5,max_notas_entre_nota_5,corr_resenhas_nota
0,1000,3.9,1.5,5.0,44,4,0.02


**Conclusão 3.2 — o livro mais resenhado não é o mais bem avaliado, e a nota média só vale quando vem de muitas notas:**

**A régua do catálogo.** A média das notas médias dos 1.000 livros é 3,9, numa escala em que a menor média é 1,5 e a maior é 5,0. Uso 3,9 como referência para dizer se um livro está bem ou mal avaliado neste serviço.

**Popularidade não é qualidade.** *Twilight* é o livro com mais resenhas (7) e mais notas (160), mas a nota média dele é 3,66, abaixo da média do catálogo. Logo atrás, com 6 resenhas cada, os livros de *Harry Potter* têm médias de 4,41 e 4,29. A correlação entre o número de resenhas e a nota média é 0,02, ou seja, praticamente nenhuma: um livro muito comentado não tem nota melhor por causa disso, podem indicar livros com debate maior, polêmicos, etc (variadas hipóteses).

**Nota 5 com poucas notas.** 44 livros (4,4%) têm nota média 5, mas nenhum deles passa de 4 notas. Com tão poucas avaliações, uma única nota alta já define a média. Isso não aponta os preferidos dos leitores, e é por isso que a tarefa 3.4 só considera livros com 50 notas ou mais.

**Para o novo produto.** O app deve mostrar a nota média junto com o número de notas, para o leitor saber quanto confiar nela. Em listas de "mais bem avaliados", vale exigir um mínimo de notas antes de ranquear. Assim um livro com 2 notas 5 não passa à frente de um com 80 notas e média 4,4. Popularidade (notas e resenhas) e qualidade percebida (nota média) são informações diferentes, e as duas ajudam o leitor a escolher.


## 3.3 Editora com mais livros de mais de 50 páginas

> *Enunciado: "Identifique a editora que publicou o maior número de livros com mais de 50 páginas (excluindo folhetos)."*

Junto `books` com `publishers` e conto os livros de cada editora com `num_pages > 50`. Uso `>` porque o enunciado diz "mais de 50", e o filtro tira os folhetos. Guardo o limite numa constante e conto à parte quantos livros ficam de fora, para mostrar o tamanho do corte.

Trago as 5 primeiras editoras, e não só a primeira, para ver se há empate no topo e qual a distância para a segunda colocada. A coluna `posicao` usa `RANK()`, que dá a mesma posição para editoras empatadas. A parcela de cada editora é calculada sobre o total de livros com mais de 50 páginas, com `SUM(...) OVER ()`, que soma a coluna inteira sem agrupar as linhas.

Conto cada `publisher_id` como ele está no banco. Na seção 2.1 vi que algumas editoras aparecem com mais de um nome (Ace, Ace Book, Ace Books, Ace Hardcover). Juntar essas variações seria uma suposição minha sobre os dados, então não faço isso e aviso na conclusão.

In [114]:
# "mais de 50 páginas": uso >, e o que fica de fora são os folhetos
MIN_PAGINAS = 50

# Acho o números de editoras e quantas delas têm livros acima do corte
# Acho também o números médio de livros por editora

folhetos = consulta(f"""
    SELECT
        COUNT(*) FILTER (WHERE num_pages <= {MIN_PAGINAS})                     AS folhetos_excluidos,
        COUNT(*) FILTER (WHERE num_pages > {MIN_PAGINAS})                      AS livros_acima_do_corte,
        COUNT(DISTINCT publisher_id)                                           AS editoras,
        COUNT(DISTINCT publisher_id) FILTER (WHERE num_pages > {MIN_PAGINAS})  AS editoras_acima_do_corte,
        ROUND(1.0 * COUNT(*) FILTER (WHERE num_pages > {MIN_PAGINAS})
                / COUNT(DISTINCT publisher_id) FILTER (WHERE num_pages > {MIN_PAGINAS}), 2) AS media_livros_por_editora
    FROM books;
""")
display(folhetos)

# RANK() dá a mesma posição a editoras empatadas; SUM() OVER () soma a coluna inteira sem agrupar
editoras = consulta(f"""
    WITH por_editora AS (
        SELECT
            publishers.publisher_id,
            publishers.publisher,
            COUNT(*) AS livros
        FROM books AS books
        JOIN publishers AS publishers ON publishers.publisher_id = books.publisher_id
        WHERE books.num_pages > {MIN_PAGINAS}
        GROUP BY publishers.publisher_id, publishers.publisher
    )
    SELECT
        RANK() OVER (ORDER BY livros DESC)             AS posicao,
        publisher,
        livros,
        ROUND(100.0 * livros / SUM(livros) OVER (), 2) AS pct_dos_livros
    FROM por_editora
    ORDER BY livros DESC
    LIMIT 5;
""")
editoras

,folhetos_excluidos,livros_acima_do_corte,editoras,editoras_acima_do_corte,media_livros_por_editora
0,8,992,340,334,2.97


,posicao,publisher,livros,pct_dos_livros
0,1,Penguin Books,42,4.23
1,2,Vintage,31,3.13
2,3,Grand Central Publishing,25,2.52
3,4,Penguin Classics,24,2.42
4,5,Ballantine Books,19,1.92


**Conclusão 3.3 — Penguin Books é a editora com mais livros de mais de 50 páginas, com 42 títulos (4,23%):**

**O corte.** Só 8 livros (0,8%) têm 50 páginas ou menos e ficam de fora como folhetos. Sobram 992 livros, de 334 das 340 editoras do banco.

**A primeira colocada.** Penguin Books tem 42 livros, 4,23% dos 992, sem empate. A Vintage vem em segundo, com 31 (3,13%), e a Grand Central Publishing em terceiro, com 25 (2,52%). A média é de 2,97 livros por editora, então a Penguin Books tem bem mais títulos do que uma editora comum do catálogo.

**Os nomes das editoras.** Conto cada `publisher_id` como está no banco. Penguin Classics, a quarta colocada com 24 livros (2,42%), aparece separada da Penguin Books. Se as duas fossem contadas juntas, o primeiro lugar não mudaria, só ficaria com mais vantagem, mantenho os nomes e a ordem como está por aceitar que pode haver editoras com nomes diferentes.

**Para o novo produto.** O catálogo é muito espalhado: mesmo a maior editora tem só 4,23% dos livros. Uma parceria com a Penguin Books é o primeiro passo natural, mas uma parceria só não basta para formar um acervo parecido com o do concorrente. O app vai precisar de acordos com várias editoras.

## 3.4 Autor com a maior nota média

> *Enunciado: "Identifique o autor com a nota média mais alta, considerando apenas livros com pelo menos 50 avaliações."*

Primeiro separo, numa CTE, os livros com pelo menos 50 notas. O filtro é `HAVING COUNT(*) >= 50`, porque o enunciado diz "pelo menos". Na seção 3.2 vi que a nota média de um livro com poucas notas não diz muita coisa (os 44 livros com média 5 têm no máximo 4 notas), e esse filtro resolve isso.

Depois junto esses livros com `books` e `authors` e calculo a nota de cada autor de duas formas:

1. `nota_media`: a média de todas as notas dos livros dele que passaram no filtro. É a resposta que uso, porque cada nota pesa igual.
2. `media_das_medias`: a média das médias de cada livro. Nela, um livro com 50 notas pesa o mesmo que um com 160.

Mostro as duas para conferir se a escolha muda o primeiro lugar. Trago também quantos livros e quantas notas sustentam a média de cada autor, e numa consulta à parte conto quantos livros e quantos autores passam no filtro.

In [115]:
# "pelo menos 50": uso >=
MIN_NOTAS = 50

# a CTE guarda a soma e a contagem das notas de cada livro; juntas elas dão a média de todas as notas do autor
sql_livros_50 = f"""
    SELECT
        book_id,
        COUNT(*)    AS n_notas,
        SUM(rating) AS soma_notas,
        AVG(rating) AS nota_media_livro
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(*) >= {MIN_NOTAS}
"""

filtro = consulta(f"""
    WITH livros_50 AS ({sql_livros_50})
    SELECT
        COUNT(*)                        AS livros_com_50_notas,
        COUNT(DISTINCT books.author_id) AS autores
    FROM livros_50 AS livros_50
    JOIN books AS books ON books.book_id = livros_50.book_id;
""")
display(filtro)

# 1.0 * evita a divisão inteira (SUM de inteiro dividido por SUM de inteiro corta as casas decimais)
autores = consulta(f"""
    WITH livros_50 AS ({sql_livros_50})
    SELECT
        authors.author,
        COUNT(*)                                                           AS livros,
        SUM(livros_50.n_notas)                                             AS notas,
        ROUND(1.0 * SUM(livros_50.soma_notas) / SUM(livros_50.n_notas), 2) AS nota_media,
        ROUND(AVG(livros_50.nota_media_livro), 2)                          AS media_das_medias
    FROM livros_50 AS livros_50
    JOIN books AS books     ON books.book_id = livros_50.book_id
    JOIN authors AS authors ON authors.author_id = books.author_id
    GROUP BY authors.author_id, authors.author
    ORDER BY nota_media DESC
    LIMIT 5;
""")
autores

,livros_com_50_notas,autores
0,19,14


,author,livros,notas,nota_media,media_das_medias
0,J.K. Rowling/Mary GrandPré,4,310.0,4.29,4.28
1,Markus Zusak/Cao Xuân Việt Khương,1,53.0,4.26,4.26
2,J.R.R. Tolkien,2,162.0,4.25,4.26
3,Louisa May Alcott,1,52.0,4.19,4.19
4,Rick Riordan,1,62.0,4.08,4.08


In [116]:
# o primeiro colocado sai da própria consulta (ORDER BY + LIMIT 1), sem digitar o nome do autor
# trago todos os livros dele, inclusive os com menos de 50 notas, para ver quais entraram na média
livros_do_primeiro = consulta(f"""
    WITH livros_50 AS ({sql_livros_50}),
    primeiro AS (
        SELECT books.author_id
        FROM livros_50 AS livros_50
        JOIN books AS books ON books.book_id = livros_50.book_id
        GROUP BY books.author_id
        ORDER BY 1.0 * SUM(livros_50.soma_notas) / SUM(livros_50.n_notas) DESC
        LIMIT 1
    ),
    notas_livro AS (
        SELECT book_id, COUNT(*) AS n_notas, ROUND(AVG(rating), 2) AS nota_media
        FROM ratings
        GROUP BY book_id
    )
    SELECT
        authors.author,
        books.title,
        books.publication_date,
        notas_livro.n_notas,
        notas_livro.nota_media,
        notas_livro.n_notas >= {MIN_NOTAS} AS passa_filtro
    FROM books AS books
    JOIN primeiro AS primeiro       ON primeiro.author_id = books.author_id
    JOIN authors AS authors         ON authors.author_id = books.author_id
    LEFT JOIN notas_livro AS notas_livro ON notas_livro.book_id = books.book_id
    ORDER BY notas_livro.n_notas DESC;
""")
livros_do_primeiro

,author,title,publication_date,n_notas,nota_media,passa_filtro
0,J.K. Rowling/Mary GrandPré,Harry Potter and the Prisoner of Azkaban (Harr...,2004-05-01,82,4.41,True
1,J.K. Rowling/Mary GrandPré,Harry Potter and the Chamber of Secrets (Harry...,1999-06-02,80,4.29,True
2,J.K. Rowling/Mary GrandPré,Harry Potter and the Order of the Phoenix (Har...,2004-09-01,75,4.19,True
3,J.K. Rowling/Mary GrandPré,Harry Potter and the Half-Blood Prince (Harry ...,2006-09-16,73,4.25,True
4,J.K. Rowling/Mary GrandPré,Harry Potter Boxed Set Books 1-5 (Harry Potte...,2004-09-13,2,4.50,False


**Conclusão 3.4 — J.K. Rowling/Mary GrandPré tem a maior nota média entre os autores com livros de pelo menos 50 notas, 4,29:**

**O filtro.** Só 19 livros (1,9% do catálogo) têm 50 notas ou mais, e eles são de 14 autores. O ranking compara poucos autores, mas cada média vem de bastante gente.

**O primeiro lugar.** J.K. Rowling/Mary GrandPré tem nota média 4,29, calculada sobre 4 livros e 310 notas. É o autor com mais livros e mais notas entre os primeiros. Markus Zusak/Cao Xuân Việt Khương vem em segundo, com 4,26 (1 livro, 53 notas), e J.R.R. Tolkien em terceiro, com 4,25 (2 livros, 162 notas). A diferença entre o primeiro e o terceiro é de 0,04 ponto.

**Os livros por trás da média.** Os 4 livros que entram no cálculo são da série Harry Potter: *Prisoner of Azkaban* (82 notas, média 4,41), *Chamber of Secrets* (80 notas, 4,29), *Half-Blood Prince* (73 notas, 4,25) e *Order of the Phoenix* (75 notas, 4,19). O autor tem um quinto livro no catálogo, o box com os livros 1 a 5, que fica de fora: tem média 4,50, mas só 2 notas. É o mesmo caso dos livros com nota 5 da seção 3.2, uma média alta que vem de pouca gente.

**A forma de calcular.** Uso a média de todas as notas do autor, em que cada nota pesa igual. Pela média das médias dos livros, Rowling fica com 4,28 e continua em primeiro, então a escolha não muda o resultado.

**Os nomes dos autores.** Conto cada `author_id` como está no banco. Os dois primeiros colocados aparecem com dois nomes juntos, separados por `/`, e não juntei nem separei esses nomes, pois temos aparentemente autores em conjunto trabalhando nos projetos.

**Para o novo produto.** Uma série inteira com nota alta e muitas avaliações, como Harry Potter, é o tipo de título que segura o leitor no app, porque suponho que quem gosta de um livro tende a seguir para o próximo. Como só 19 livros passam de 50 notas, o concorrente tem poucos títulos com avaliação confiável. Um app que incentive o leitor a dar nota pode construir essa base mais rápido.

## 3.5 Média de resenhas de quem avaliou mais de 50 livros

> *Enunciado: "Encontre o número médio de resenhas com texto entre usuários que avaliaram mais de 50 livros."*

A resposta precisa de dois níveis de agregação. Primeiro conto, em `ratings`, quantos livros cada usuário avaliou. Depois conto, em `reviews`, quantas resenhas com texto cada usuário escreveu. Faço cada contagem numa CTE separada e só depois junto as duas pelo `username`, para não multiplicar as linhas (seção 2.2).

Três decisões:

1. "Mais de 50 livros" vira `> 50`, contando livros distintos (`COUNT(DISTINCT book_id)`), para o caso de um usuário ter avaliado o mesmo livro duas vezes.
2. "Com texto" vira `text` não nulo e não vazio. Antes, confiro quantas resenhas ficariam de fora por esse filtro.
3. Usuário que avaliou mais de 50 livros e não escreveu nenhuma resenha entra na média com zero. Ele faz parte do grupo que o enunciado pede, e deixar ele de fora puxaria a média para cima. Para isso uso `LEFT JOIN` e `COALESCE`.

Para ter com o que comparar, calculo a mesma média para os demais usuários, os que avaliaram 50 livros ou menos. A consulta devolve uma linha para cada grupo.

In [117]:
# "mais de 50 livros": uso >
MIN_LIVROS_AVALIADOS = 50

# quantas resenhas o filtro "com texto" tira: texto nulo ou só com espaços
textos = consulta("""
    SELECT
        COUNT(*)                                                    AS resenhas,
        COUNT(*) FILTER (WHERE text IS NULL OR TRIM(text) = '')     AS resenhas_sem_texto
    FROM reviews;
""")
display(textos)

# o banco não tem tabela de usuários: o total sai da união dos username de ratings e reviews
# UNION (sem ALL) tira os nomes repetidos, então quem está nas duas tabelas conta uma vez só
usuarios = consulta("""
    SELECT
        (SELECT COUNT(DISTINCT username) FROM ratings) AS usuarios_com_nota,
        (SELECT COUNT(DISTINCT username) FROM reviews) AS usuarios_com_resenha,
        COUNT(*)                                       AS usuarios_no_total
    FROM (
        SELECT username FROM ratings
        UNION
        SELECT username FROM reviews
    ) AS todos;
""")
display(usuarios)

# cada CTE conta uma tabela por usuário; o LEFT JOIN mantém quem avaliou mas não escreveu resenha
# o CASE dá nome aos dois grupos; o PostgreSQL aceita o alias grupo sozinho 
# no GROUP BY e no ORDER BY
resenhas_por_grupo = consulta(f"""
    WITH avaliadores AS (
        SELECT username, COUNT(DISTINCT book_id) AS livros_avaliados
        FROM ratings
        GROUP BY username
    ),
    resenhas AS (
        SELECT username, COUNT(*) AS n_resenhas
        FROM reviews
        WHERE text IS NOT NULL AND TRIM(text) <> ''
        GROUP BY username
    )
    SELECT
        CASE WHEN avaliadores.livros_avaliados > {MIN_LIVROS_AVALIADOS}
            THEN 'mais de {MIN_LIVROS_AVALIADOS} livros'
            ELSE '{MIN_LIVROS_AVALIADOS} livros ou menos'
        END                                                   AS grupo,
        COUNT(*)                                              AS usuarios,
        ROUND(AVG(COALESCE(resenhas.n_resenhas, 0)), 2)       AS media_resenhas,
        MIN(COALESCE(resenhas.n_resenhas, 0))                 AS min_resenhas,
        MAX(COALESCE(resenhas.n_resenhas, 0))                 AS max_resenhas,
        COUNT(*) FILTER (WHERE resenhas.n_resenhas IS NULL)   AS usuarios_sem_resenha,
        ROUND(AVG(avaliadores.livros_avaliados), 2)           AS media_livros_avaliados
    FROM avaliadores AS avaliadores
    LEFT JOIN resenhas AS resenhas ON resenhas.username = avaliadores.username
    GROUP BY grupo
    ORDER BY grupo DESC;
""")
resenhas_por_grupo

,resenhas,resenhas_sem_texto
0,2793,0


,usuarios_com_nota,usuarios_com_resenha,usuarios_no_total
0,160,160,160


,grupo,usuarios,media_resenhas,min_resenhas,max_resenhas,usuarios_sem_resenha,media_livros_avaliados
0,mais de 50 livros,6,24.33,18,28,0,54.33
1,50 livros ou menos,154,17.19,5,29,0,39.81


**Conclusão 3.5 — os 6 usuários que avaliaram mais de 50 livros escrevem, em média, 24,33 resenhas:**

**O grupo.** Só 6 dos 160 usuários avaliaram mais de 50 livros. Cada um escreveu entre 18 e 28 resenhas, e a média é 24,33. O filtro "com texto" do enunciado não tira nenhuma resenha, porque as 2.793 têm texto.

**A comparação.** Os outros 154 usuários escrevem 17,19 resenhas em média, cerca de 7 a menos. Mas os grupos se misturam: o usuário que mais escreveu (29 resenhas) avaliou 50 livros ou menos, e nenhum usuário da base ficou sem resenha. Com só 6 pessoas no grupo dos mais ativos, a diferença indica uma tendência, não uma regra.

**Para o novo produto.** Quem avalia muitos livros também escreve bastante, e todos os usuários da base escrevem pelo menos algumas resenhas. Vale investir em resenhas e comunidade no app, e os leitores mais ativos são os primeiros a convidar para isso, talvez algo que desenvolva o hábito de fazer mais resenhas e debates na comunidade do app, por exemplo (ideia que poderia ser atrativa).

---

# 4. Conclusão geral

1. **O acervo do concorrente é recente e espalhado.** 81,9% dos livros saíram após 01/01/2000, e nenhuma editora tem mais de 4,23% do catálogo (seções 3.1 e 3.3).
2. **Nota média só é confiável com muitas avaliações, e popularidade não é qualidade.** Só 19 livros têm 50 notas ou mais, e a correlação entre número de resenhas e nota média é 0,02 (seções 3.2 e 3.4).
3. **Os leitores escrevem.** Todos os 160 usuários da base deram pelo menos uma nota e escreveram pelo menos uma resenha, e os 6 que avaliaram mais de 50 livros escrevem 24,33 em média, contra 17,19 dos demais (seção 3.5).

**Os dados.** O banco tem 1.000 livros, 6.456 notas e 2.793 resenhas, com `books` no centro e as outras tabelas ligadas a ele (seção 2.1). Todas as chaves têm par, e 994 livros (99,4%) têm pelo menos uma resenha. Juntar `ratings` e `reviews` direto gera 26.167 linhas, então em todas as tarefas agreguei cada tabela antes da junção (seção 2.2).

**O acervo.** 819 dos 1.000 livros (81,9%) foram publicados após 01/01/2000, e as datas vão de 1952 a 2020 (seção 3.1). Excluindo 8 folhetos, os 992 livros restantes vêm de 334 editoras, com média de 2,97 livros cada. A primeira colocada é a Penguin Books, com 42 livros (4,23%), seguida pela Vintage, com 31 (3,13%) (seção 3.3).

**Notas e resenhas.** A média das notas médias do catálogo é 3,9 (seção 3.2). O livro mais resenhado, *Twilight* (7 resenhas, 160 notas), tem média 3,66, abaixo dessa régua. 44 livros (4,4%) têm média 5, mas nenhum passa de 4 notas (seção 3.2). Com o mínimo de 50 notas, sobram 19 livros de 14 autores, e o primeiro é J.K. Rowling/Mary GrandPré, com 4,29 em 4 livros de Harry Potter e 310 notas. Os três primeiros autores ficam a 0,04 ponto um do outro (seção 3.4).

**Os leitores.** O banco não tem uma tabela de usuários. Contando os nomes de `ratings` e `reviews` juntos, a base tem 160 usuários, e todos aparecem nas duas tabelas: não há quem só dê nota nem quem só escreva resenha. Os 6 que avaliaram mais de 50 livros escrevem entre 18 e 28 resenhas. Os outros 154 escrevem 17,19 em média, e o usuário que mais escreveu (29) está nesse segundo grupo (seção 3.5).

**Limitações.**
- A base é pequena e parece gerada artificialmente: os textos das resenhas não fazem sentido (seção 2.1), e são só 160 usuários, todos com pelo menos uma nota e uma resenha (seção 3.5). Suponho que num serviço real haja bem mais gente que só dá nota do que gente que escreve (os dados não mostram isso), então os números descrevem esta base, não necessariamente o serviço real.
- Os grupos comparados são pequenos (6 usuários na seção 3.5, 19 livros na seção 3.4), e não fiz teste estatístico. As diferenças são tendências.
- Editoras e autores não têm nome padronizado. A mesma editora pode aparecer com nomes diferentes (porém aceito que sejam distintas também), e um autor pode ser um grupo de coautores (seções 2.1, 3.3 e 3.4). Contei cada id como está no banco.
- Os leitores mais ativos também avaliam mais livros (54,33 contra 39,81 em média), então parte da diferença de resenhas pode vir só de lerem mais. Não testei essa hipótese (seção 3.5).
- Os dados vão até 31/03/2020 (seção 3.1).

**Recomendações para o novo app.**
1. **Montar um acervo atual com várias editoras, começando pela Penguin Books.** O catálogo do concorrente é recente e nenhuma editora concentra mais de 4,23% dele, então uma parceria só não basta.
2. **Mostrar a nota média sempre junto com o número de notas, e exigir um mínimo de notas nos rankings.** Sem isso, um livro com 2 notas 5 passa à frente de um com 82 notas e média 4,41, por exemplo.
3. **Separar "mais comentados" de "mais bem avaliados".** As duas coisas não andam juntas (correlação de 0,02), e cada lista ajuda o leitor de um jeito.
4. **Destacar séries com nota alta e muitas avaliações, como Harry Potter.** São os títulos com avaliação mais confiável do catálogo.
5. **Investir em resenhas e comunidade, começando pelos leitores mais ativos.** Todos os 160 usuários escrevem pelo menos uma resenha, e os mais ativos escrevem mais. Incentivar notas também resolve o problema de só 19 livros terem avaliação confiável.